# Airbnb Market Pricing Model: Multiple Linear Regression

**Objective:** Engineer features and train a scalable Linear Regression model to estimate Airbnb rental prices. The pipeline processes multiple numerical and categorical inputs—including property attributes, neighborhood data, and review scores—to generate accurate price predictions.

**Key Techniques:** VectorAssembler, multiple feature regression, RMSE, and R-Squared evaluation.

### Step 1 & 2: Setup, Load Data, and Explore

In [1]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

# Initialize Spark
spark = SparkSession.builder.appName("Assignment3_PricePrediction").getOrCreate()

# Load the CSV file
df = spark.read.csv("airbnb_price_prediction_sample (1).csv", header=True, inferSchema=True)

# Show the schema and the first 5 rows
df.printSchema()
df.show(5)

root
 |-- id: integer (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- bedrooms: double (nullable = true)
 |-- beds: double (nullable = true)
 |-- accommodates: integer (nullable = true)
 |-- minimum_nights: integer (nullable = true)
 |-- number_of_reviews: integer (nullable = true)
 |-- review_scores_rating: double (nullable = true)
 |-- property_type: string (nullable = true)
 |-- room_type: string (nullable = true)
 |-- neighborhood: string (nullable = true)
 |-- price: double (nullable = true)

+---+---------+--------+----+------------+--------------+-----------------+--------------------+-------------+------------+---------------+------+
| id|bathrooms|bedrooms|beds|accommodates|minimum_nights|number_of_reviews|review_scores_rating|property_type|   room_type|   neighborhood| price|
+---+---------+--------+----+------------+--------------+-----------------+--------------------+-------------+------------+---------------+------+
|  1|      1.0|     1.0| 2.0|           

### Step 3: Split the Data (80% Training, 20% Testing)

In [2]:
trainDF, testDF = df.randomSplit([0.8, 0.2], seed=42)

print(f"Training Data Count: {trainDF.count()}")
print(f"Testing Data Count: {testDF.count()}")

Training Data Count: 98
Testing Data Count: 22


### Step 4: Prepare Features (Adding EXTRA Features!)
We are using the 5 features suggested in the assignment, PLUS `minimum_nights` and `number_of_reviews` to improve the model even more.

In [3]:
from pyspark.ml.feature import VectorAssembler

# Added minimum_nights and number_of_reviews for better accuracy
input_columns = [
    "bathrooms", 
    "bedrooms", 
    "beds", 
    "accommodates", 
    "review_scores_rating",
    "minimum_nights",      
    "number_of_reviews"    
]

vecAssembler = VectorAssembler(
    inputCols=input_columns,
    outputCol="features"
)

# Transform the training data
vecTrainDF = vecAssembler.transform(trainDF)
vecTrainDF.select("features", "price").show(5, truncate=False)

+--------------------------------+------+
|features                        |price |
+--------------------------------+------+
|[1.0,1.0,2.0,3.0,74.2,2.0,57.0] |146.65|
|[3.0,1.0,1.0,2.0,85.2,2.0,59.0] |252.6 |
|[1.0,1.0,1.0,2.0,72.9,1.0,97.0] |168.71|
|[4.0,3.0,3.0,7.0,78.8,1.0,141.0]|431.81|
|[1.0,1.0,2.0,2.0,95.7,3.0,20.0] |195.22|
+--------------------------------+------+
only showing top 5 rows


### Step 5: Build and Train the Linear Regression Model

In [4]:
from pyspark.ml.regression import LinearRegression

# Configure the Linear Regression model
lr = LinearRegression(
    featuresCol="features", 
    labelCol="price",
    predictionCol="prediction"
)

# Train the model using the training data
lrModel = lr.fit(vecTrainDF)

print("Model successfully trained!")

Model successfully trained!


### Step 6: Make Predictions on the Test Data
Let's see how our model predicts the price on data it has never seen before.

In [5]:
# Transform the testing data using the assembler
vecTestDF = vecAssembler.transform(testDF)

# Generate predictions
predTestDF = lrModel.transform(vecTestDF)

# Show Actual Price vs Predicted Price
predTestDF.select("price", "prediction", "features").show(15)

+------+------------------+--------------------+
| price|        prediction|            features|
+------+------------------+--------------------+
|274.16|  302.001023929037|[3.0,1.0,1.0,4.0,...|
|227.21|237.68117998883878|[2.5,1.0,1.0,3.0,...|
|149.63|176.61813320738213|[1.0,1.0,1.0,3.0,...|
|319.03|318.51103185433476|[4.0,1.0,2.0,4.0,...|
|362.57|  322.504729938254|[1.0,3.0,4.0,6.0,...|
|255.15|270.28443449520796|[4.0,1.0,1.0,2.0,...|
|217.86|236.98916041682128|[2.0,1.0,2.0,4.0,...|
|445.83|437.92797324954137|[4.0,3.0,4.0,7.0,...|
|336.57|339.19109112140654|[3.0,2.0,2.0,6.0,...|
|362.99|383.42221360179917|[1.0,3.0,4.0,7.0,...|
|302.99|327.78352417427743|[2.5,2.0,2.0,5.0,...|
|452.07| 467.1894544380627|[4.0,3.0,3.0,7.0,...|
|284.39|267.11657512378696|[4.0,1.0,2.0,2.0,...|
|494.39|511.25469299301704|[3.0,4.0,4.0,10.0...|
| 169.1|168.62819566160243|[1.0,1.0,1.0,2.0,...|
+------+------------------+--------------------+
only showing top 15 rows


### Step 7: Evaluate Model Performance
Using RMSE (Root Mean Squared Error) and R2 (Accuracy) to see how well our extra features performed.

In [6]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator_rmse = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(predTestDF)
r2 = evaluator_r2.evaluate(predTestDF)

print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R-Squared (R2) Model Accuracy: {r2:.2f}")

Root Mean Squared Error (RMSE): 17.63
R-Squared (R2) Model Accuracy: 0.96
